# Spine XR Augmentation — Colab Runner

Bu notebook Google Colab Pro+ (A100) üzerinde çalışacak şekilde tasarlandı. Tüm ağır eğitim buradan koşturulur.

## Sıralama

1. Bootstrap (Drive mount, repo clone, pip install, StyleGAN2-ADA clone)
2. `01_audit` → `02_splits`
3. `03_train_classifier --config baseline.yaml --fold {1,2,3}` + `03b_summarize_cv`
4. `03_train_classifier --config traditional.yaml --fold {1,2,3}` + `03b_summarize_cv`
5. StyleGAN: `04a_prepare_gan_pool` → `04b_train_stylegan --stage base` → `--stage per_class --class <name>` × 7 → `04c_generate_stylegan`
6. LDM: `06_train_ldm --config vae.yaml` → `06_train_ldm --config ldm.yaml` → `07_generate_ldm`
7. `03_train_classifier --config stylegan_aug.yaml --fold {1,2,3}` + `ldm_aug.yaml` + `combined.yaml`
8. `09_test_eval --configs ...` → `10_make_report`

## 1. Bootstrap

In [ ]:
# 1. Drive Mount
from google.colab import drive
drive.mount('/content/drive')

import os
import shutil
from pathlib import Path

# 2. Çalışma Alanını Yerel SSD'de Ayarla (A100'ün maksimum hızı için)
LOCAL_ROOT = Path('/content/spine-xr-augmentation-study')
LOCAL_ROOT.mkdir(parents=True, exist_ok=True)
os.chdir(LOCAL_ROOT)

# 3. Kodları ve Ağırlıkları Drive'dan Yerele Kopyala
DRIVE_REPO_PATH = Path('/content/drive/MyDrive/spine-xr/spine-xr-augmentation-study')
!cp -r {DRIVE_REPO_PATH}/* .

# 4. Dataset'i SSD'ye Çek ve Aç (Dataset Drive'da .rar olarak durmalı)
# Klasör Adını "final_preprocessed_data" Yerine "dataset" Yap
DRIVE_DATASET_PATH = Path('/content/drive/MyDrive/spine-xr/dataset.rar') 
!unrar x {DRIVE_DATASET_PATH} {LOCAL_ROOT}/

# 5. Çıktıların (Outputs) Kaybolmaması İçin Drive'a Bağla
DRIVE_OUTPUTS = Path('/content/drive/MyDrive/spine-xr/outputs')
DRIVE_OUTPUTS.mkdir(parents=True, exist_ok=True)

if os.path.exists('outputs'):
    shutil.rmtree('outputs')
os.symlink(DRIVE_OUTPUTS, 'outputs')

print(f"Çalışma dizini (SSD): {os.getcwd()}")
!ls -l

In [ ]:
!pip install -q -r requirements.txt
!python -c "import torch; print(torch.__version__, torch.cuda.is_available(), torch.cuda.get_device_name(0))"

In [ ]:
# StyleGAN2-ADA NVIDIA repo clone
!mkdir -p third_party && cd third_party && [ -d stylegan2-ada-pytorch ] || git clone https://github.com/NVlabs/stylegan2-ada-pytorch.git

## 2. Audit + Splits

In [ ]:
!python scripts/01_audit.py --config configs/base.yaml
!python scripts/02_splits.py --config configs/base.yaml
!cat outputs/01_audit/audit_report.md

## 3. Baseline classifier — 3 fold

In [ ]:
# SMOKE test önce (3 epoch fold 1)
!python scripts/03_train_classifier.py --config configs/classifier/baseline.yaml --fold 1 --smoke

In [ ]:
for k in (1, 2, 3):
    !python scripts/03_train_classifier.py --config configs/classifier/baseline.yaml --fold {k}
!python scripts/03b_summarize_cv.py --config configs/classifier/baseline.yaml

## 4. Traditional

In [ ]:
for k in (1, 2, 3):
    !python scripts/03_train_classifier.py --config configs/classifier/traditional.yaml --fold {k}
!python scripts/03b_summarize_cv.py --config configs/classifier/traditional.yaml

## 5. StyleGAN2-ADA

In [ ]:
!python scripts/04a_prepare_gan_pool.py
!python scripts/04b_train_stylegan.py --stage base
CLASSES = ['Osteophytes','Disc_space_narrowing','Other_lesions','Foraminal_stenosis','Surgical_implant','Spondylolysthesis','Vertebral_collapse']
for c in CLASSES:
    !python scripts/04b_train_stylegan.py --stage per_class --class "{c.replace('_', ' ')}"
!python scripts/04c_generate_stylegan.py

## 6. Latent Diffusion

In [ ]:
!python scripts/06_train_ldm.py --config configs/diffusion/vae.yaml
!python scripts/06_train_ldm.py --config configs/diffusion/ldm.yaml
!python scripts/07_generate_ldm.py

## 7. Augmented classifier runs

In [ ]:
!python scripts/08_build_combined_metadata.py
for cfg in ('stylegan_aug', 'ldm_aug', 'combined'):
    for k in (1, 2, 3):
        !python scripts/03_train_classifier.py --config configs/classifier/{cfg}.yaml --fold {k}
    !python scripts/03b_summarize_cv.py --config configs/classifier/{cfg}.yaml

## 8. Test evaluation + Final report

In [ ]:
!python scripts/09_test_eval.py --configs configs/classifier/baseline.yaml configs/classifier/traditional.yaml configs/classifier/stylegan_aug.yaml configs/classifier/ldm_aug.yaml configs/classifier/combined.yaml
!python scripts/10_make_report.py --config configs/base.yaml
!cat outputs/final_report.md | head -200